In [5]:
from statsbombpy import sb
import pandas as pd

competitions = sb.competitions()

serie_a = competitions[competitions['competition_name'] == 'Serie A']
serie_a

c:\Users\madha\xG-model-football\xG-model-football\venv\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
70,12,27,Italy,Serie A,male,False,False,2015/2016,2025-08-15T14:28:50.169562,NaN,NaN,2025-08-15T14:28:50.169562
71,12,86,Italy,Serie A,male,False,False,1986/1987,2025-11-23T11:00:00.442491,NaN,NaN,2025-11-23T11:00:00.442491


In [7]:
matches = sb.matches(competition_id=12, season_id=27)
print(f"Total matches: {len(matches)}")
matches.head()
matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']]

c:\Users\madha\xG-model-football\xG-model-football\venv\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total matches: 380


,match_id,match_date,home_team,away_team,home_score,away_score
0,3879608,2015-11-07,Hellas Verona,Bologna,0,2
1,3879551,2015-09-27,Hellas Verona,Lazio,1,2
2,3879575,2015-10-18,Hellas Verona,Udinese,1,1
3,3879542,2015-09-23,Inter Milan,Hellas Verona,1,0
4,3879600,2015-11-01,Carpi,Hellas Verona,0,0
...,...,...,...,...,...,...
375,3878545,2015-08-23,Sampdoria,Carpi,5,2
376,3878544,2015-08-23,Palermo,Genoa,1,0
377,3878543,2015-08-23,Inter Milan,Atalanta,1,0
378,3878542,2015-08-23,Fiorentina,AC Milan,2,0


In [8]:
events = sb.events(match_id=3878542)
print(f"Total events: {len(events)}")
events['type'].value_counts()

c:\Users\madha\xG-model-football\xG-model-football\venv\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total events: 3483


type
Pass               974
Ball Receipt*      943
Carry              731
Pressure           360
Ball Recovery       97
Duel                47
Block               40
Foul Committed      36
Foul Won            36
Miscontrol          29
Goal Keeper         27
Clearance           26
Dribble             23
Shot                23
Interception        20
Dispossessed        17
Dribbled Past       15
Shield               6
Substitution         6
Half Start           4
Half End             4
50/50                4
Injury Stoppage      3
Starting XI          2
Error                2
Bad Behaviour        2
Tactical Shift       2
Player Off           2
Player On            2
Name: count, dtype: int64

In [ ]:
#separating x and y and defining shots
shots = events[events['type'] == 'Shot'].copy()

shots['x'] = shots['location'].apply(lambda loc: loc[0])
shots['y'] = shots['location'].apply(lambda loc: loc[1])

shots[['minute', 'team', 'player', 'x', 'y', 'shot_body_part', 'shot_type', 'shot_outcome']].head(10)

,minute,team,player,x,y,shot_body_part,shot_type,shot_outcome
3412,5,Fiorentina,Josip Iličić,88.0,74.8,Right Foot,Open Play,Saved
3413,10,Fiorentina,Josip Iličić,90.3,32.2,Left Foot,Free Kick,Off T
3414,13,AC Milan,Nigel de Jong,94.1,40.4,Right Foot,Open Play,Blocked
3415,13,AC Milan,Giacomo Bonaventura,107.1,31.3,Left Foot,Open Play,Blocked
3416,14,AC Milan,Carlos Arturo Bacca Ahumada,103.7,51.7,Right Foot,Open Play,Off T
3417,15,Fiorentina,Marcos Alonso Mendoza,90.9,22.2,Left Foot,Open Play,Wayward
3418,16,Fiorentina,Milan Badelj,95.2,43.7,Right Foot,Open Play,Blocked
3419,19,Fiorentina,Nikola Kalinić,102.7,36.8,Right Foot,Open Play,Saved
3420,19,Fiorentina,Josip Iličić,100.1,43.5,Left Foot,Open Play,Blocked
3421,20,Fiorentina,Marcos Alonso Mendoza,112.0,21.4,Left Foot,Open Play,Off T


In [ ]:
import numpy as np
#calculating distance to goal
#According to StatBomb, the pitch co-ordinates run from (0,0) to (120, 80) [length, width]
#The goal the attack team would be aiming to score would then be located at (120,40) since the goal is usually in the middle of the pitch and at the furthest point away from you length wise.
#hence we can define it like this:
goal_x, goal_y = 120, 40

shots['distance_to_goal'] = np.sqrt((goal_x - shots['x'])**2 + (goal_y - shots['y'])**2)

shots[['x', 'y', 'distance_to_goal']].head(10)

,x,y,distance_to_goal
3412,88.0,74.8,47.276210
3413,90.3,32.2,30.707165
3414,94.1,40.4,25.903089
3415,107.1,31.3,15.559563
3416,103.7,51.7,20.064396
3417,90.9,22.2,34.112314
3418,95.2,43.7,25.074489
3419,102.7,36.8,17.593465
3420,100.1,43.5,20.205445
3421,112.0,21.4,20.247469


In [14]:
# Now we need to calculate angle
#According to StatBomb the goal is 8 units wide. This means 4 on either side from the center which is 40 so we can define it as:
left_post_y, right_post_y = 36, 44

angle_left = np.arctan2(left_post_y - shots['y'], goal_x - shots['x'])
angle_right = np.arctan2(right_post_y - shots['y'], goal_x - shots['x'])

shots['angle_to_goal'] = np.abs(angle_left - angle_right)

shots[['x', 'y', 'distance_to_goal', 'angle_to_goal']].head(10)

,x,y,distance_to_goal,angle_to_goal
3412,88.0,74.8,47.276210,0.114857
3413,90.3,32.2,30.707165,0.250927
3414,94.1,40.4,25.903089,0.306389
3415,107.1,31.3,15.559563,0.428193
3416,103.7,51.7,20.064396,0.325332
3417,90.9,22.2,34.112314,0.200134
3418,95.2,43.7,25.074489,0.313143
3419,102.7,36.8,17.593465,0.440590
3420,100.1,43.5,20.205445,0.385542
3421,112.0,21.4,20.247469,0.161046
